# Nouvelle route IA Ask Toby / Mianatra — `tenant_id + message` vers réponse intelligente

Ce notebook ajoute une **nouvelle fonctionnalité API** au-dessus de la base MySQL `tenant_management`.

Objectif : pour chaque requête utilisateur :

```json
{
  "tenant_id": 6,
  "message": "Résumer les derniers échanges et préparer le prochain suivi."
}
```

Le pipeline fait :

1. **LLM 1 Gemini = filtre / routeur IA** : il classe le message dans un type précis.
2. **Récupération SQL ciblée** : on récupère seulement les tables utiles selon le type.
3. **LLM 2 Gemini = réponse finale** : il synthétise les données récupérées et répond en texte.
4. **Route FastAPI** : endpoint `POST /tenant/prompt`.

Les catégories prévues :

| Catégorie | Quand l'utiliser |
|---|---|
| `follow_up_exchange` | Derniers échanges, appels, formations, assistances, prochain suivi, choses à faire |
| `missing_documents` | Documents reçus, absents, en attente, validés ou non reçus |
| `interaction_detail` | Résumé détaillé de toutes les interactions faites avec le tenant |
| `progression` | Questions sur les stages 1 à 5 et l'avancement |
| `tenant_general` | Question générale sur le tenant : « comment va ce tenant ? », « qu'est-ce qui se passe ? » |

> Important : ce notebook utilise les mêmes tables que ton projet : `tenants`, `calls`, `formation_assistance`, `tenant_interactions`, `data_items`, `progression_tracker`, `users`, etc.

## Recherches / choix techniques pour faciliter l'implémentation

Pour cette fonctionnalité, le plus simple est de faire une **classification structurée** avec Gemini avant de récupérer les données SQL.

Choix retenus :

- **`google-genai`** : SDK Python officiel pour utiliser Gemini.
- **Structured output avec Pydantic** : permet au premier Gemini de répondre en JSON propre, par exemple `{category, confidence, reason}`.
- **FastAPI + Pydantic** : pratique pour créer une route `POST` avec un body `{tenant_id, message}` et une réponse contrôlée.
- **SQLAlchemy + PyMySQL** : connexion propre à MySQL local depuis Python/Jupyter.
- **Pandas** : rend les résultats SQL plus faciles à transformer en dictionnaires pour le contexte LLM.

Architecture :

```text
POST /tenant/prompt
        |
        v
[tenant_id, message]
        |
        v
Gemini #1 : classifier le message
        |
        +--> follow_up_exchange  -> calls + formation_assistance + tenant_interactions
        +--> missing_documents   -> data_items + data_item_attachments
        +--> interaction_detail  -> calls + formation_assistance + tenant_interactions
        +--> progression         -> progression_tracker + data_items + formation_assistance
        +--> tenant_general      -> synthèse globale des 4 types
        |
        v
Gemini #2 : générer la réponse finale en français
        |
        v
{ category, answer, context_summary }
```

In [ ]:
# ============================================================
# 1) Installer les dépendances
# ============================================================
# À exécuter une fois dans ton environnement Jupyter / VS Code.

%pip install -q google-genai sqlalchemy pymysql pandas python-dotenv pydantic fastapi uvicorn nest-asyncio

## 2) Configuration `.env`

Crée un fichier `.env` à côté du notebook avec tes vraies informations :

```env
MYSQL_HOST=127.0.0.1
MYSQL_PORT=3306
MYSQL_USER=root
MYSQL_PASSWORD=ton_mot_de_passe_mysql
MYSQL_DATABASE=tenant_management

GEMINI_API_KEY=ta_cle_api_gemini
GEMINI_CLASSIFIER_MODEL=gemini-2.5-flash
GEMINI_ANSWER_MODEL=gemini-2.5-flash
```

`127.0.0.1` est souvent plus stable que `localhost` avec MySQL local.

In [ ]:
# ============================================================
# 3) Imports et chargement de configuration
# ============================================================

from __future__ import annotations

import os
import json
import math
from datetime import date, datetime
from enum import Enum
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from sqlalchemy import create_engine, text as sa_text
from sqlalchemy.engine import URL

from google import genai
from google.genai import types

load_dotenv()

MYSQL_HOST = os.getenv("MYSQL_HOST", "127.0.0.1")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "tenant_management")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_CLASSIFIER_MODEL = os.getenv("GEMINI_CLASSIFIER_MODEL", "gemini-2.5-flash")
GEMINI_ANSWER_MODEL = os.getenv("GEMINI_ANSWER_MODEL", "gemini-2.5-flash")

print("Base MySQL:", MYSQL_DATABASE)
print("Gemini configuré:", bool(GEMINI_API_KEY))

In [ ]:
# ============================================================
# 4) Connexion MySQL locale
# ============================================================

mysql_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE,
    query={"charset": "utf8mb4"},
)

engine = create_engine(mysql_url, pool_pre_ping=True, future=True)


def read_sql(sql: str, params: Optional[dict] = None) -> pd.DataFrame:
    """Exécute une requête SELECT et retourne un DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql_query(sa_text(sql), conn, params=params or {})


def test_connection() -> None:
    df = read_sql("SELECT DATABASE() AS database_name, NOW() AS server_time")
    display(df)


test_connection()

In [ ]:
# ============================================================
# 5) Utilitaires de nettoyage
# ============================================================

NULL_TEXT = "non défini"


def clean_value(value: Any) -> Any:
    """Convertit les valeurs SQL/Pandas en valeurs propres pour JSON/LLM."""
    if value is None:
        return NULL_TEXT
    if isinstance(value, float) and math.isnan(value):
        return NULL_TEXT
    try:
        if pd.isna(value):
            return NULL_TEXT
    except Exception:
        pass
    if isinstance(value, (datetime, date, pd.Timestamp)):
        return str(value)[:19]
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="ignore")
    return value


def clean_record(record: Dict[str, Any]) -> Dict[str, Any]:
    return {k: clean_value(v) for k, v in record.items()}


def df_to_records(df: pd.DataFrame) -> List[Dict[str, Any]]:
    return [clean_record(row) for row in df.to_dict(orient="records")]


def is_defined(value: Any) -> bool:
    return value not in (None, "", NULL_TEXT) and not (isinstance(value, float) and math.isnan(value))


def normalize_status(text: Any) -> str:
    if not is_defined(text):
        return ""
    return str(text).strip().lower().replace("é", "e").replace("è", "e").replace("ê", "e")


def compact_json(obj: Any, max_chars: int = 16000) -> str:
    raw = json.dumps(obj, ensure_ascii=False, indent=2, default=str)
    if len(raw) <= max_chars:
        return raw
    return raw[:max_chars] + "\n... [contexte tronqué pour éviter un prompt trop long]"

In [ ]:
# ============================================================
# 6) Modèles Pydantic : requête API, classification, réponse API
# ============================================================

class RouteCategory(str, Enum):
    follow_up_exchange = "follow_up_exchange"
    missing_documents = "missing_documents"
    interaction_detail = "interaction_detail"
    progression = "progression"
    tenant_general = "tenant_general"


class TenantPromptRequest(BaseModel):
    tenant_id: int = Field(..., description="ID du tenant dans la table tenants")
    message: str = Field(..., min_length=1, description="Question ou instruction de l'utilisateur")


class RouteDecision(BaseModel):
    category: RouteCategory
    confidence: float = Field(..., ge=0, le=1)
    reason: str
    needs_global_synthesis: bool = False


class TenantPromptResponse(BaseModel):
    tenant_id: int
    tenant_name: str
    category: RouteCategory
    confidence: float
    answer: str
    context_used: Dict[str, Any]

## 7) Fonctions SQL ciblées

On ne donne pas toute la base au LLM. On récupère uniquement les tables nécessaires selon la catégorie.

Règle importante : les clés étrangères comme `tenant_id`, `created_by`, `called_by` sont remplacées par des informations lisibles : nom du tenant, code tenant, nom de l'utilisateur.

In [ ]:
# ============================================================
# 7.1 Tenant profile
# ============================================================

def fetch_tenant_profile(tenant_id: int) -> Dict[str, Any]:
    df = read_sql(
        """
        SELECT
            t.*,
            u.name AS created_by_name,
            u.email AS created_by_email
        FROM tenants t
        LEFT JOIN users u ON u.id = t.created_by
        WHERE t.id = :tenant_id
        LIMIT 1
        """,
        {"tenant_id": tenant_id},
    )
    records = df_to_records(df)
    if not records:
        raise ValueError(f"Tenant introuvable pour tenant_id={tenant_id}")
    return records[0]


# Test rapide : change l'ID si besoin
# fetch_tenant_profile(6)

In [ ]:
# ============================================================
# 7.2 Calls / appels
# ============================================================

def fetch_calls(tenant_id: int, limit: Optional[int] = None) -> List[Dict[str, Any]]:
    limit_sql = "LIMIT :limit" if limit else ""
    params = {"tenant_id": tenant_id}
    if limit:
        params["limit"] = limit

    df = read_sql(
        f"""
        SELECT
            c.id,
            c.tenant_id,
            t.name AS tenant_name,
            t.code AS tenant_code,
            c.caller_name,
            c.interlocutor_name,
            c.called_by,
            u.name AS called_by_name,
            c.call_date,
            c.call_time,
            c.duration_seconds,
            c.call_type,
            c.subject,
            c.details,
            c.follow_up_required,
            c.follow_up_date,
            c.created_at,
            c.updated_at
        FROM calls c
        LEFT JOIN tenants t ON t.id = c.tenant_id
        LEFT JOIN users u ON u.id = c.called_by
        WHERE c.tenant_id = :tenant_id
        ORDER BY c.call_date DESC, c.call_time DESC, c.id DESC
        {limit_sql}
        """,
        params,
    )
    return df_to_records(df)


def extract_call_followups(calls: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    followups = []
    for c in calls:
        required = c.get("follow_up_required") in (1, True, "1")
        follow_date = c.get("follow_up_date")
        if required or is_defined(follow_date):
            followups.append({
                "source": "calls",
                "call_id": c.get("id"),
                "date": c.get("follow_up_date"),
                "subject": c.get("subject"),
                "details": c.get("details"),
                "interlocutor": c.get("interlocutor_name"),
                "status": "suivi requis" if required else "suivi mentionné",
            })
    return followups

In [ ]:
# ============================================================
# 7.3 Formations / assistances
# ============================================================

def fetch_formations_assistances(tenant_id: int, limit: Optional[int] = None) -> List[Dict[str, Any]]:
    limit_sql = "LIMIT :limit" if limit else ""
    params = {"tenant_id": tenant_id}
    if limit:
        params["limit"] = limit

    df = read_sql(
        f"""
        SELECT
            fa.id,
            fa.tenant_id,
            t.name AS tenant_name,
            t.code AS tenant_code,
            fa.type,
            fa.portal,
            fa.location,
            fa.scheduled_date,
            fa.scheduled_time,
            fa.duration_hours,
            fa.completion_date,
            fa.completion_time,
            fa.actual_duration_hours,
            fa.description,
            fa.feedback,
            fa.google_drive_link,
            fa.trainer_name,
            fa.satisfaction_form_uploaded,
            fa.photo_uploaded,
            fa.notes,
            fa.status,
            fa.roles,
            fa.participants_present,
            fa.participants_roles,
            fa.real_duration_hours,
            fa.user_feedbacks,
            fa.feedback_document_url,
            fa.reference_link,
            fa.completion_notes,
            fa.created_by,
            u.name AS created_by_name,
            fa.created_at,
            fa.updated_at
        FROM formation_assistance fa
        LEFT JOIN tenants t ON t.id = fa.tenant_id
        LEFT JOIN users u ON u.id = fa.created_by
        WHERE fa.tenant_id = :tenant_id
        ORDER BY COALESCE(fa.scheduled_date, fa.completion_date, DATE(fa.created_at)) DESC, fa.id DESC
        {limit_sql}
        """,
        params,
    )
    return df_to_records(df)


def extract_formation_followups(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    followups = []
    for item in items:
        status = normalize_status(item.get("status"))
        item_type = item.get("type", "formation/assistance")
        is_done = status in {"completed", "termine", "terminee", "done", "closed", "cloture", "annule", "cancelled"}
        has_schedule = is_defined(item.get("scheduled_date"))
        has_completion = is_defined(item.get("completion_date"))
        if (not is_done) or (has_schedule and not has_completion and status not in {"cancelled", "annule"}):
            followups.append({
                "source": "formation_assistance",
                "id": item.get("id"),
                "type": item_type,
                "date": item.get("scheduled_date"),
                "location": item.get("location"),
                "description": item.get("description"),
                "trainer_name": item.get("trainer_name"),
                "status": item.get("status"),
                "notes": item.get("notes"),
            })
    return followups

In [ ]:
# ============================================================
# 7.4 Tenant interactions : meetings, visites, suivis, etc.
# ============================================================

def fetch_tenant_interactions(tenant_id: int, limit: Optional[int] = None) -> List[Dict[str, Any]]:
    limit_sql = "LIMIT :limit" if limit else ""
    params = {"tenant_id": tenant_id}
    if limit:
        params["limit"] = limit

    df = read_sql(
        f"""
        SELECT
            ti.id,
            ti.tenant_id,
            t.name AS tenant_name,
            t.code AS tenant_code,
            ti.interaction_type,
            ti.title,
            ti.interaction_date,
            ti.interaction_time,
            ti.location,
            ti.mianatra_team,
            ti.tenant_representatives,
            ti.situation,
            ti.discussed_points,
            ti.needs_identified,
            ti.decisions,
            ti.next_actions,
            ti.follow_up_date,
            ti.status,
            ti.priority,
            ti.created_by,
            uc.name AS created_by_name,
            ti.updated_by,
            uu.name AS updated_by_name,
            ti.created_at,
            ti.updated_at
        FROM tenant_interactions ti
        LEFT JOIN tenants t ON t.id = ti.tenant_id
        LEFT JOIN users uc ON uc.id = ti.created_by
        LEFT JOIN users uu ON uu.id = ti.updated_by
        WHERE ti.tenant_id = :tenant_id
          AND ti.deleted_at IS NULL
        ORDER BY ti.interaction_date DESC, ti.interaction_time DESC, ti.id DESC
        {limit_sql}
        """,
        params,
    )
    return df_to_records(df)


def extract_interaction_followups(interactions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    followups = []
    for i in interactions:
        status = normalize_status(i.get("status"))
        if is_defined(i.get("follow_up_date")) or is_defined(i.get("next_actions")) or status in {"open", "pending", "en cours"}:
            followups.append({
                "source": "tenant_interactions",
                "interaction_id": i.get("id"),
                "title": i.get("title"),
                "date": i.get("follow_up_date"),
                "next_actions": i.get("next_actions"),
                "decisions": i.get("decisions"),
                "status": i.get("status"),
                "priority": i.get("priority"),
            })
    return followups

In [ ]:
# ============================================================
# 7.5 Documents / data items
# ============================================================

DEFAULT_REQUIRED_DOCUMENTS = [
    "Brochure ou présentation de l'établissement",
    "Liste des classes",
    "Liste des élèves",
    "Liste des responsables ou parents",
    "Liste du personnel / enseignants",
    "Grille tarifaire / frais scolaires",
    "Emplois du temps",
    "Photos / logo / images de l'établissement",
    "Informations de domaine, DNS ou accès portail",
]

VALID_DOCUMENT_STATUSES = {
    "valide", "validé", "validee", "validée", "validated",
    "recu", "reçu", "received", "ok", "complet", "complete",
}


def fetch_data_items(tenant_id: int) -> List[Dict[str, Any]]:
    """Récupère les documents demandés/reçus pour un tenant.

    Règle demandée : si status est NULL, 'En attente', ou pas validé => non reçu.
    """
    df = read_sql(
        """
        SELECT
            di.id,
            di.tenant_id,
            t.name AS tenant_name,
            t.code AS tenant_code,
            di.title,
            di.description,
            di.is_default,
            di.status,
            di.notes,
            di.created_by,
            u.name AS created_by_name,
            di.created_at,
            di.updated_at,
            COUNT(dia.id) AS attachment_count,
            GROUP_CONCAT(DISTINCT dia.file_name SEPARATOR ' | ') AS attachment_names,
            GROUP_CONCAT(DISTINCT dia.link_url SEPARATOR ' | ') AS attachment_links
        FROM data_items di
        LEFT JOIN tenants t ON t.id = di.tenant_id
        LEFT JOIN users u ON u.id = di.created_by
        LEFT JOIN data_item_attachments dia ON dia.data_item_id = di.id
        WHERE di.tenant_id = :tenant_id
        GROUP BY di.id
        ORDER BY di.created_at DESC, di.id DESC
        """,
        {"tenant_id": tenant_id},
    )
    items = df_to_records(df)
    for item in items:
        status_norm = normalize_status(item.get("status"))
        item["receipt_status"] = "reçu" if status_norm in VALID_DOCUMENT_STATUSES else "non reçu"
        item["rule_explanation"] = "validé donc reçu" if item["receipt_status"] == "reçu" else "pas validé, NULL ou en attente donc non reçu"
    return items


def summarize_document_state(items: List[Dict[str, Any]]) -> Dict[str, Any]:
    received = [x for x in items if x.get("receipt_status") == "reçu"]
    missing = [x for x in items if x.get("receipt_status") != "reçu"]
    return {
        "required_baseline": DEFAULT_REQUIRED_DOCUMENTS,
        "documents_recorded_in_database": items,
        "received_documents": received,
        "missing_or_not_validated_documents": missing,
        "rule": "Un document est considéré comme reçu seulement si son statut est validé/reçu/ok/complet. NULL ou En attente = non reçu.",
    }

In [ ]:
# ============================================================
# 7.6 Progression tracker : stages 1 à 5
# ============================================================

STAGE_LABELS = {
    "stage1": "Stage 1 — Signature du contrat",
    "stage2": "Stage 2 — DNS / domaine / propagation",
    "stage3": "Stage 3 — Collecte des données et documents",
    "stage4": "Stage 4 — Saisie des données",
    "stage5": "Stage 5 — Formation, satisfaction et handover",
}

STAGE_NEXT_ACTIONS = {
    "stage1": "Vérifier la signature du contrat et les informations de domaine.",
    "stage2": "Finaliser le domaine, DNS ou la propagation.",
    "stage3": "Collecter et valider les documents manquants.",
    "stage4": "Terminer la saisie des données dans le système.",
    "stage5": "Terminer la formation, récupérer le formulaire de satisfaction et faire le handover.",
}


def fetch_progression(tenant_id: int) -> Dict[str, Any]:
    df = read_sql(
        """
        SELECT
            p.*,
            t.name AS tenant_name,
            t.code AS tenant_code
        FROM progression_tracker p
        LEFT JOIN tenants t ON t.id = p.tenant_id
        WHERE p.tenant_id = :tenant_id
        LIMIT 1
        """,
        {"tenant_id": tenant_id},
    )
    records = df_to_records(df)
    if not records:
        return {"progression_found": False, "message": "Aucune ligne progression_tracker trouvée pour ce tenant."}

    p = records[0]
    stages = []

    def completed(field: str) -> bool:
        return p.get(field) in (1, True, "1")

    stages.append({"stage": "stage1", "label": STAGE_LABELS["stage1"], "completed": completed("stage1_completed"), "details": {"domain_name": p.get("stage1_domain_name")}})
    stages.append({"stage": "stage2", "label": STAGE_LABELS["stage2"], "completed": completed("stage2_completed"), "details": {"dns_creation_date": p.get("stage2_dns_creation_date"), "dns_passation_date": p.get("stage2_dns_passation_date")}})
    stages.append({"stage": "stage3", "label": STAGE_LABELS["stage3"], "completed": completed("stage3_completed"), "details": {}})
    stages.append({"stage": "stage4", "label": STAGE_LABELS["stage4"], "completed": completed("stage4_completed"), "details": {"data_entry_date_start": p.get("stage4_data_entry_date_start"), "data_entry_date_end": p.get("stage4_data_entry_date_end"), "data_types": p.get("stage4_data_types")}})

    stage5_completed = completed("stage5_completed") and completed("stage5_formations_finished") and completed("stage5_satisfaction_form_received") and completed("stage5_handover_completed")
    stages.append({"stage": "stage5", "label": STAGE_LABELS["stage5"], "completed": stage5_completed, "details": {"stage5_completed": p.get("stage5_completed"), "formations_finished": p.get("stage5_formations_finished"), "satisfaction_form_received": p.get("stage5_satisfaction_form_received"), "handover_completed": p.get("stage5_handover_completed"), "handover_date": p.get("stage5_handover_date")}})

    current = next((s for s in stages if not s["completed"]), stages[-1])
    all_done = all(s["completed"] for s in stages)

    return {
        "progression_found": True,
        "raw": p,
        "stages": stages,
        "current_stage": "Terminé" if all_done else current["label"],
        "next_action": "Tous les stages semblent terminés." if all_done else STAGE_NEXT_ACTIONS[current["stage"]],
    }

## 8) Gemini #1 : filtre IA / classification du message

Le premier Gemini ne répond pas à l'utilisateur. Son rôle est seulement de décider quelle logique SQL utiliser.

Exemples :

- « Résumer les derniers échanges et préparer le prochain suivi » → `follow_up_exchange`
- « Quels documents sont encore absents ? » → `missing_documents`
- « Résume toutes les interactions faites avec ce tenant » → `interaction_detail`
- « On est à quel stage ? » → `progression`
- « Comment va ce tenant ? » → `tenant_general`

In [ ]:
# ============================================================
# 8) Client Gemini + classification structurée
# ============================================================

client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

CLASSIFIER_SYSTEM_PROMPT = """
Tu es un routeur IA pour une application de gestion de tenants scolaires.
Tu dois classer le message utilisateur dans UNE seule catégorie.

Catégories disponibles :
1. follow_up_exchange : derniers échanges, appels, formations, assistances, prochain suivi, prochaines actions, relance, choses à faire.
2. missing_documents : documents, fichiers, pièces, documents reçus, documents absents, documents en attente, documents validés, documents nécessaires.
3. interaction_detail : résumé détaillé de toutes les interactions, appels, réunions, visites, formations, assistances déjà effectuées.
4. progression : stage 1 à 5, étape actuelle, avancement, progression, handover, DNS, collecte, data entry, formation.
5. tenant_general : question générale sur le tenant, son état global, ce qui se passe avec lui. Cette catégorie doit synthétiser les autres.

Règles :
- Si le message demande les derniers échanges + prochain suivi, choisis follow_up_exchange.
- Si le message parle de documents manquants, reçus, validés ou nécessaires, choisis missing_documents.
- Si le message demande toutes les interactions sans insister sur le prochain suivi, choisis interaction_detail.
- Si le message parle de stage, stade, étape ou progression, choisis progression.
- Si le message est vague/général comme "comment va ce tenant ?", choisis tenant_general.
""".strip()


def local_rule_classifier(message: str) -> RouteDecision:
    """Fallback local si la clé Gemini n'est pas configurée."""
    m = message.lower()
    if any(w in m for w in ["document", "fichier", "pièce", "piece", "absent", "reçu", "recu", "valid", "en attente"]):
        category = RouteCategory.missing_documents
    elif any(w in m for w in ["stage", "stade", "étape", "etape", "progression", "avancement", "handover", "dns", "data entry"]):
        category = RouteCategory.progression
    elif any(w in m for w in ["prochain", "suivi", "relance", "à faire", "a faire", "future", "échange", "echange", "derniers"]):
        category = RouteCategory.follow_up_exchange
    elif any(w in m for w in ["interaction", "appel", "formation", "assistance", "réunion", "reunion", "visite"]):
        category = RouteCategory.interaction_detail
    else:
        category = RouteCategory.tenant_general
    return RouteDecision(category=category, confidence=0.55, reason="Fallback local par mots-clés", needs_global_synthesis=category == RouteCategory.tenant_general)


def classify_message_with_gemini(message: str) -> RouteDecision:
    """Gemini #1 : classe le message dans une catégorie exploitable par le pipeline SQL."""
    if client is None:
        return local_rule_classifier(message)

    prompt = f"""
{CLASSIFIER_SYSTEM_PROMPT}

Message utilisateur : {message!r}

Retourne uniquement la classification structurée.
""".strip()

    response = client.models.generate_content(
        model=GEMINI_CLASSIFIER_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=RouteDecision,
        ),
    )

    parsed = getattr(response, "parsed", None)
    if isinstance(parsed, RouteDecision):
        return parsed

    data = json.loads(response.text)
    return RouteDecision(**data)


# Test de classification
for q in [
    "Résumer les derniers échanges et préparer le prochain suivi.",
    "Quels documents sont encore absents ?",
    "On est en quel stage ?",
    "Résume toutes les interactions faites avec ce tenant.",
    "Comment va ce tenant ?",
]:
    print(q, "=>", classify_message_with_gemini(q).model_dump())

## 9) Construire le contexte selon la catégorie

Cette étape évite de donner trop de données au LLM final.

- `follow_up_exchange` → derniers échanges + prochains suivis.
- `missing_documents` → data items + règle reçu/non reçu.
- `interaction_detail` → timeline complète des interactions.
- `progression` → stages 1 à 5 + documents/formations si utiles.
- `tenant_general` → synthèse globale des 4 catégories.

In [ ]:
# ============================================================
# 9) Construction de contexte ciblé
# ============================================================

def build_follow_up_exchange_context(tenant_id: int) -> Dict[str, Any]:
    tenant = fetch_tenant_profile(tenant_id)
    calls = fetch_calls(tenant_id, limit=10)
    interactions = fetch_tenant_interactions(tenant_id, limit=10)
    formations = fetch_formations_assistances(tenant_id, limit=15)

    followups = []
    followups.extend(extract_call_followups(calls))
    followups.extend(extract_interaction_followups(interactions))
    followups.extend(extract_formation_followups(formations))

    return {
        "intent": "Résumer les derniers échanges et préparer le prochain suivi.",
        "tenant": tenant,
        "recent_calls": calls,
        "recent_tenant_interactions": interactions,
        "recent_formations_assistances": formations,
        "detected_followups_or_next_actions": followups,
    }


def build_missing_documents_context(tenant_id: int) -> Dict[str, Any]:
    tenant = fetch_tenant_profile(tenant_id)
    items = fetch_data_items(tenant_id)
    return {
        "intent": "Identifier les documents reçus, non reçus, en attente ou nécessaires.",
        "tenant": tenant,
        "document_state": summarize_document_state(items),
    }


def build_interaction_detail_context(tenant_id: int) -> Dict[str, Any]:
    tenant = fetch_tenant_profile(tenant_id)
    return {
        "intent": "Résumer en détail toutes les interactions faites avec le tenant.",
        "tenant": tenant,
        "calls": fetch_calls(tenant_id),
        "tenant_interactions": fetch_tenant_interactions(tenant_id),
        "formations_assistances": fetch_formations_assistances(tenant_id),
    }


def build_progression_context(tenant_id: int) -> Dict[str, Any]:
    tenant = fetch_tenant_profile(tenant_id)
    return {
        "intent": "Répondre aux questions sur la progression stage 1 à stage 5.",
        "tenant": tenant,
        "progression": fetch_progression(tenant_id),
        "documents_context_for_stage3": summarize_document_state(fetch_data_items(tenant_id)),
        "formations_context_for_stage5": fetch_formations_assistances(tenant_id, limit=10),
    }


def build_tenant_general_context(tenant_id: int) -> Dict[str, Any]:
    return {
        "intent": "Synthèse globale du tenant à partir des échanges, documents, interactions et progression.",
        "follow_up_exchange": build_follow_up_exchange_context(tenant_id),
        "missing_documents": build_missing_documents_context(tenant_id),
        "progression": build_progression_context(tenant_id),
    }


def build_context_for_category(tenant_id: int, category: RouteCategory) -> Dict[str, Any]:
    if category == RouteCategory.follow_up_exchange:
        return build_follow_up_exchange_context(tenant_id)
    if category == RouteCategory.missing_documents:
        return build_missing_documents_context(tenant_id)
    if category == RouteCategory.interaction_detail:
        return build_interaction_detail_context(tenant_id)
    if category == RouteCategory.progression:
        return build_progression_context(tenant_id)
    if category == RouteCategory.tenant_general:
        return build_tenant_general_context(tenant_id)
    raise ValueError(f"Catégorie inconnue : {category}")


# Test : construire un contexte pour tenant_id=6
# ctx = build_context_for_category(6, RouteCategory.follow_up_exchange)
# print(compact_json(ctx, max_chars=4000))

## 10) Gemini #2 : génération de la réponse finale

Le deuxième Gemini reçoit la question, la catégorie et le contexte SQL ciblé. Il répond uniquement avec les données disponibles.

In [ ]:
# ============================================================
# 10) Génération de réponse finale avec Gemini #2
# ============================================================

ANSWER_SYSTEM_PROMPT = """
Tu es l'assistant Ask Toby pour Mianatra.
Tu réponds en français clair, utile, professionnel et humain.

Règles obligatoires :
- Réponds uniquement avec les données du CONTEXTE SQL fourni.
- Si une information est absente, dis "non défini" ou "la base ne contient pas cette information".
- Ne devine pas.
- Remplace les IDs par les noms déjà fournis dans le contexte.
- Pour les documents : NULL, En attente, pending, ou statut non validé = document non reçu.
- Pour les suivis : distingue les échanges déjà faits et les prochaines actions.
- Pour la progression : explique le stage actuel, les stages terminés, les blocages et la prochaine étape.
- Pour tenant_general : synthétise état global, documents, progression, interactions et prochain suivi.

Format conseillé :
1. Réponse courte directe.
2. Détails utiles en puces.
3. Prochaines actions suggérées si pertinent.
""".strip()


def local_answer_fallback(message: str, category: RouteCategory, context: Dict[str, Any]) -> str:
    """Fallback simple si Gemini n'est pas configuré. Utile pour tester le SQL."""
    tenant = context.get("tenant") or {}
    tenant_name = tenant.get("name", NULL_TEXT)
    if category == RouteCategory.tenant_general:
        tenant_name = context.get("follow_up_exchange", {}).get("tenant", {}).get("name", NULL_TEXT)
    return (
        f"Réponse fallback locale pour le tenant {tenant_name}.\n\n"
        f"Catégorie détectée : {category.value}.\n"
        f"Gemini n'est pas configuré, donc voici le contexte SQL brut à analyser :\n\n"
        f"{compact_json(context, max_chars=5000)}"
    )


def generate_final_answer_with_gemini(message: str, category: RouteCategory, context: Dict[str, Any]) -> str:
    """Gemini #2 : produit la réponse finale en texte."""
    if client is None:
        return local_answer_fallback(message, category, context)

    prompt = f"""
{ANSWER_SYSTEM_PROMPT}

QUESTION UTILISATEUR :
{message}

CATÉGORIE DÉTECTÉE :
{category.value}

CONTEXTE SQL :
{compact_json(context, max_chars=22000)}

Réponds maintenant à la question de l'utilisateur.
""".strip()

    response = client.models.generate_content(
        model=GEMINI_ANSWER_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2),
    )
    return response.text.strip()

## 11) Pipeline complet : `tenant_id + message` → réponse

Cette fonction représente le cœur de la nouvelle route.

In [ ]:
# ============================================================
# 11) Pipeline principal
# ============================================================

def answer_tenant_prompt(tenant_id: int, message: str, include_context: bool = True) -> TenantPromptResponse:
    # 1) Vérifier que le tenant existe.
    tenant = fetch_tenant_profile(tenant_id)

    # 2) Gemini #1 : classification.
    decision = classify_message_with_gemini(message)

    # 3) Récupération SQL ciblée.
    context = build_context_for_category(tenant_id, decision.category)

    # 4) Gemini #2 : réponse finale.
    answer = generate_final_answer_with_gemini(message, decision.category, context)

    return TenantPromptResponse(
        tenant_id=tenant_id,
        tenant_name=str(tenant.get("name", NULL_TEXT)),
        category=decision.category,
        confidence=decision.confidence,
        answer=answer,
        context_used=context if include_context else {"hidden": True},
    )


# Exemple d'appel direct dans le notebook
# response = answer_tenant_prompt(
#     tenant_id=6,
#     message="Résumer les derniers échanges et préparer le prochain suivi.",
#     include_context=False,
# )
# print(response.model_dump_json(indent=2))
# print("\nRéponse finale :\n", response.answer)

## 12) Exemples de questions à tester

Change `tenant_id` selon tes données.

In [ ]:
# ============================================================
# 12) Exemples de tests
# ============================================================

TEST_TENANT_ID = 6  # Exemple : L'Ile des Enfants si tes insertions sont identiques.

example_questions = [
    "Résumer les derniers échanges et préparer le prochain suivi.",
    "Quels documents sont encore absents ou non validés ?",
    "Résume toutes les interactions faites avec ce tenant.",
    "On est en quel stage et qu'est-ce qu'il reste à faire ?",
    "Comment va ce tenant ? Qu'est-ce qui se passe avec lui ?",
]

# Décommente pour tester :
# for q in example_questions:
#     print("=" * 100)
#     print("QUESTION:", q)
#     result = answer_tenant_prompt(TEST_TENANT_ID, q, include_context=False)
#     print("CATÉGORIE:", result.category, "CONF:", result.confidence)
#     print(result.answer)

## 13) Nouvelle route FastAPI : `POST /tenant/prompt`

La route attend :

```json
{
  "tenant_id": 6,
  "message": "Résumer les derniers échanges et préparer le prochain suivi."
}
```

Elle retourne une réponse texte avec la catégorie détectée.

In [ ]:
# ============================================================
# 13) FastAPI app
# ============================================================

from fastapi import FastAPI, HTTPException

app = FastAPI(
    title="Ask Toby Tenant AI Pipeline",
    version="1.0.0",
    description="Route IA tenant_id + message avec Gemini classifier + Gemini answer.",
)


@app.get("/health")
def health_check():
    return {"status": "ok", "database": MYSQL_DATABASE, "gemini_configured": bool(GEMINI_API_KEY)}


@app.post("/tenant/prompt", response_model=TenantPromptResponse)
def tenant_prompt_route(payload: TenantPromptRequest):
    try:
        return answer_tenant_prompt(
            tenant_id=payload.tenant_id,
            message=payload.message,
            include_context=False,  # Mets True en debug si tu veux voir le contexte SQL.
        )
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erreur pipeline IA: {e}")


# Pour lancer depuis le notebook :
# import nest_asyncio
# import uvicorn
# nest_asyncio.apply()
# uvicorn.run(app, host="127.0.0.1", port=8000)

## 14) Tester la route avec `curl`

Après avoir lancé le serveur FastAPI :

```bash
curl -X POST "http://127.0.0.1:8000/tenant/prompt"   -H "Content-Type: application/json"   -d "{"tenant_id": 6, "message": "Résumer les derniers échanges et préparer le prochain suivi."}"
```

Ou avec Python :

In [ ]:
# Exemple de test HTTP avec requests, une fois le serveur lancé.
# %pip install -q requests
# import requests
# r = requests.post(
#     "http://127.0.0.1:8000/tenant/prompt",
#     json={"tenant_id": 6, "message": "Résumer les derniers échanges et préparer le prochain suivi."},
# )
# print(r.status_code)
# print(json.dumps(r.json(), ensure_ascii=False, indent=2))

## 15) Notes d'amélioration pour la production

Pour passer ce notebook dans ton projet FastAPI réel :

1. Déplacer les fonctions SQL dans `repositories/tenant_ai_repository.py`.
2. Déplacer `classify_message_with_gemini` et `generate_final_answer_with_gemini` dans `services/tenant_ai_service.py`.
3. Garder les modèles Pydantic dans `schemas/tenant_ai.py`.
4. Ajouter la route dans `routes/tenant_ai.py`.
5. Mettre `include_context=False` en production pour ne pas exposer tout le contexte SQL à l'utilisateur.
6. Ajouter des logs : catégorie détectée, confidence, durée SQL, durée Gemini.
7. Ajouter un fallback : si `confidence < 0.60`, utiliser `tenant_general` ou demander une précision.

Ce notebook est donc une version claire pour apprendre et valider le pipeline avant de l'intégrer dans ton backend.